# Phase 1 — Clean Project Foundation

## Goal

Create the reusable project structure, validate the approved Phase 0 decisions,
run automated safety tests, and write a reproducible run manifest.

This notebook **does not read or process the source corpus**. It writes only
inside `Devoteam_AI_CLEAN_PIPELINE`.

### Safe rerun behavior

- The package checksum is verified before extraction.
- Matching existing files are skipped.
- A different existing file causes a stop instead of being overwritten.
- The original source shortcut is never opened.

## Setup

Mount Drive in Colab and define the approved project boundary.

In [ ]:
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/Devoteam internship/Devoteam_AI_CLEAN_PIPELINE")
    PACKAGE_PATH = PROJECT_ROOT / "PHASE_1_FOUNDATION_PACKAGE.zip"
else:
    PROJECT_ROOT = Path(os.environ.get("DEVOTEAM_LOCAL_PROJECT_ROOT", "local_validation_project")).resolve()
    PACKAGE_PATH = Path(os.environ.get("DEVOTEAM_PHASE1_PACKAGE", "PHASE_1_FOUNDATION_PACKAGE.zip")).resolve()
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_PACKAGE_SHA256 = "57e1a3972ec737cf78603bb287b1e6c5f4b53fb84a6ea24600f376aaf985ef21"
SOURCE_SHORTCUT_ID = os.environ.get(
    "DEVOTEAM_SOURCE_SHORTCUT_ID", "REQUIRED_LOCAL_SOURCE_SHORTCUT_ID"
)  # recorded only; never opened here

print(f"Mode: {'Google Colab' if IN_COLAB else 'local validation'}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Foundation package: {PACKAGE_PATH}")

## Step 1 — Verify boundaries and package integrity

In [ ]:
assert PROJECT_ROOT.exists(), f"Project root does not exist: {PROJECT_ROOT}"
assert PACKAGE_PATH.is_file(), f"Foundation package is missing: {PACKAGE_PATH}"
if IN_COLAB:
    assert (PROJECT_ROOT / "PHASE_0_PROJECT_CHARTER.md").is_file(), (
        "The approved Phase 0 charter is missing from the clean project root"
    )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

actual_package_sha256 = sha256_file(PACKAGE_PATH)
assert actual_package_sha256 == EXPECTED_PACKAGE_SHA256, (
    "Foundation package checksum mismatch. Stop and obtain the approved package."
)

with zipfile.ZipFile(PACKAGE_PATH) as archive:
    members = archive.infolist()
    unsafe = [
        member.filename
        for member in members
        if member.filename.startswith("/") or ".." in Path(member.filename).parts
    ]
    assert not unsafe, f"Unsafe archive paths detected: {unsafe}"

print(f"PASS: package verified ({len(members)} entries)")
print(f"SHA-256: {actual_package_sha256}")

## Step 2 — Install the foundation idempotently

In [ ]:
created_files = []
skipped_identical = []
conflicts = []

with zipfile.ZipFile(PACKAGE_PATH) as archive:
    for member in archive.infolist():
        if member.is_dir():
            continue
        destination = (PROJECT_ROOT / member.filename).resolve()
        try:
            destination.relative_to(PROJECT_ROOT.resolve())
        except ValueError as error:
            raise RuntimeError(f"Archive would write outside project root: {member.filename}") from error

        incoming = archive.read(member)
        if destination.exists():
            existing_hash = hashlib.sha256(destination.read_bytes()).hexdigest()
            incoming_hash = hashlib.sha256(incoming).hexdigest()
            if existing_hash == incoming_hash:
                skipped_identical.append(member.filename)
            else:
                conflicts.append(member.filename)

assert not conflicts, (
    "Different existing foundation files were found. Nothing was overwritten. "
    f"Conflicts: {conflicts}"
)

with zipfile.ZipFile(PACKAGE_PATH) as archive:
    for member in archive.infolist():
        if member.is_dir() or member.filename in skipped_identical:
            continue
        destination = (PROJECT_ROOT / member.filename).resolve()
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)
        created_files.append(member.filename)

print(f"Created files: {len(created_files)}")
print(f"Skipped identical files: {len(skipped_identical)}")
print("PASS: no existing file was overwritten")

## Step 3 — Check the minimal dependency

In [ ]:
if importlib.util.find_spec("yaml") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements" / "base.txt")],
        check=True,
    )
else:
    import yaml
    print(f"PyYAML already available: {yaml.__version__}")

## Step 4 — Validate configuration and create lineage

In [ ]:
environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src")

validation = subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "validate_foundation.py"),
        "--project-root",
        str(PROJECT_ROOT),
    ],
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)
print(validation.stdout)
if validation.stderr:
    print(validation.stderr)
assert validation.returncode == 0, "Foundation validation failed"
validation_result = json.loads(validation.stdout)
assert validation_result["status"] == "PASS"

## Step 5 — Run automated safety tests

In [ ]:
tests = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)
print(tests.stdout)
print(tests.stderr)
assert tests.returncode == 0, "One or more Phase 1 tests failed"
assert "Ran 9 tests" in tests.stderr or "Ran 9 tests" in tests.stdout

## Step 6 — Write the Phase 1 release record

In [ ]:
release_record = {
    "schema_version": 1,
    "stage": "phase1_foundation_release",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "mode": "colab" if IN_COLAB else "local_validation",
    "project_root": str(PROJECT_ROOT),
    "package_sha256": actual_package_sha256,
    "created_file_count": len(created_files),
    "skipped_identical_count": len(skipped_identical),
    "automated_test_count": 9,
    "source_processed": False,
    "source_mutated": False,
    "external_llm_called": False,
    "phase_zero_decisions": ["A1", "B1", "C1", "D1", "E1", "F1", "G1"],
    "validation_manifest": validation_result["manifest"],
    "status": "PASS",
}

release_path = PROJECT_ROOT / "manifests" / "runs" / (
    "PHASE1_RELEASE_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".json"
)
release_path.write_text(
    json.dumps(release_record, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(f"Release record: {release_path}")

## Checks

The final gate must pass before Phase 2 can begin.

In [ ]:
required_paths = [
    "config/project.yaml",
    "config/security.yaml",
    "config/models.yaml",
    "config/filters.yaml",
    "src/devoteam_reference_ai/config.py",
    "tests/test_security.py",
    "docs/PHASE_1_RUNBOOK.md",
]
missing = [path for path in required_paths if not (PROJECT_ROOT / path).is_file()]
assert not missing, f"Required foundation files are missing: {missing}"
assert release_record["status"] == "PASS"
assert release_record["source_processed"] is False
assert release_record["source_mutated"] is False
assert release_record["external_llm_called"] is False

print("=" * 72)
print("PHASE 1 FOUNDATION: PASS")
print("9 automated tests passed")
print("Source corpus processed: NO")
print("Source corpus modified: NO")
print("External LLM called: NO")
print(f"Run manifest: {validation_result['manifest']}")
print(f"Release record: {release_path}")
print("=" * 72)

## Next steps

After the final `PASS`, preserve the run manifest and release record. Phase 2
may then begin with a read-only Drive inventory and immutable snapshot design.